In [2]:
# Import required libraries
import warnings
import math
import statistics
import urllib.request
from csv import DictReader
import solas_disparity as sd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import statsmodels.api as sm
from patsy import dmatrices
from scipy.stats import chi2

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from statsmodels.tools.sm_exceptions import PerfectSeparationError
from lifelines import CoxTimeVaryingFitter, KaplanMeierFitter
from lime.lime_tabular import LimeTabularExplainer
import dice_ml
from dice_ml import Dice

warnings.filterwarnings('ignore')

In [3]:
# Shared model objects are defined once and reused in the explainability section below.
lr_pipeline = None
gbt_pipeline = None
black_idx = None
white_idx = None


### Step 1. Load + prepare COMPAS data

Preparing the Numeric_features & Category_features as in Lecture 01.

In [4]:
def build_compas_model_df(raw_data, target, features):
    """Prepare the explainability pipeline dataset used in the final section of the notebook."""
    # Process the raw data to create the features and target variable for the model, 
    # including creating the same categorical factors with the same reference levels as in the R code, and 
    compas_df = prepare_compas_subset(raw_data) 

    # Make binary outcome
    compas_df[target] = (compas_df["score_factor"] == "HighScore").astype(int)

    # We will create a model dataframe that includes the features and target variable, and we will drop any rows with missing values in the target variable to ensure we have a clean dataset for modeling.
    compas_model_df = compas_df[features + [target]].dropna(subset=[target]).copy()
    
    return compas_model_df

In [5]:
# -----------------------------
# 1. Load + prepare COMPAS data
# -----------------------------
# Use the same feature set as the logistic regression model
features = [
    "gender_factor",
    "age_factor",
    "race_factor",
    "priors_count",
    "crime_factor",
    "two_year_recid",
]
# We will keep these as numeric features for the model, and we will handle them with imputation and scaling in the preprocessor.
    
numeric_features = ["priors_count", "two_year_recid"] 

 # We will use the original categorical variables for one-hot encoding in the preprocessor.
categorical_features = [
    "gender_factor",
    "age_factor",
    "race_factor",
    "crime_factor",
]

# We will use "score_binary" as the target variable for the model, which is a binary variable indicating whether the score is HighScore or not.
target = "score_binary"

# Load the raw COMPAS data from the URL and prepare the model dataframe using the build_compas_model_df function defined above, 
# which will create the features and target variable with the same factor coding as in the R code.
compas_model_df = build_compas_model_df(raw_data, target, features)

print("------------ Data Preparation Summary -----------------------------------")
print("\nThe compas_model_df dataframe has been prepared with the following characteristics:")
print("Shape of the processed dataframe for modeling:", compas_model_df.shape)
print("Columns in the model dataframe:", compas_model_df.columns.tolist())
print("Rows in model dataframe:", len(compas_model_df))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target variable:", target)
print("All features:", features)
print("-------------------------------------------------------------------------")
print("\nSample of the prepared model dataframe:") 
compas_model_df.head()

NameError: name 'raw_data' is not defined

### Step 2 — Train / test split

In [ ]:
# -----------------------------
# 2. Train / test split
# -----------------------------
X = compas_model_df[features].copy()
y = compas_model_df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Compute counts
n_total = len(X)
n_train = len(X_train)
n_test = len(X_test)

# Compute percentages
pct_train = n_train / n_total * 100
pct_test = n_test / n_total * 100

print(
    f"Train: {X_train.shape} ({n_train:,} rows, {pct_train:.1f}%), "
    f"Test: {X_test.shape} ({n_test:,} rows, {pct_test:.1f}%)"
)

### Step 3. Preprocessing pipelines

In [ ]:
# -----------------------------
# 3. Preprocessing pipelines
# -----------------------------

def build_preprocessor(numeric_features, categorical_features):
    """Build a ColumnTransformer preprocessor for numeric and categorical features, including imputation and scaling/encoding steps."""
    # For numeric features, we will impute missing values with the median and then apply standard scaling.
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    # For categorical features, we will impute missing values with the most frequent category and then apply one-hot encoding.
    # We set handle_unknown="ignore" in OneHotEncoder to avoid errors if there are categories in the test set that were not seen during training.
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])

preprocessor = build_preprocessor(numeric_features, categorical_features)
preprocessor


### Step 4 — Fit logistic regression and gradient-boosted tree

In [ ]:
# -----------------------------
# 4. Full modeling pipeline
# -----------------------------

# Logistic regression (interpretable baseline)
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

lr_pipeline.fit(X_train, y_train)

# Gradient Boosted Trees (black-box model): less interpretable / stronger nonlinear model
gbt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        random_state=42
    )),
])

gbt_pipeline.fit(X_train, y_train)


In [ ]:
# -----------------------------
# 5. Evaluation
# -----------------------------

def evaluate_classifier(model, X_test, y_test):
    # Evaluate the classifier on the test set and print various performance metrics, including accuracy, ROC AUC, confusion matrix, and classification report.
    y_pred = model.predict(X_test) # We will use the predict method to get the predicted class labels for the test set.
    y_prob = model.predict_proba(X_test)[:, 1] # We will use the predict_proba method to get the predicted probabilities for the positive class (class 1) for the test set.
    print("\nAccuracy:")
    print(accuracy_score(y_test, y_pred))
    print("\nROC AUC:")
    print(roc_auc_score(y_test, y_prob))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    return y_pred, y_prob

y_pred, y_prob = evaluate_classifier(lr_pipeline, X_test, y_test)
# -----------------------------
# 6. Optional: inspect feature names
# -----------------------------
feature_names = lr_pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Number of transformed features:", len(feature_names))
print(feature_names[:20])

# Parts A–E: Automated analysis and explanations

This final section runs Part A through Part E from the assignment outline: distribution drift checks, generalization diagnostics, a spurious-correlation probe (counterfactual swaps), a robustness sweep on `priors_count` with ICE-style summaries, and slice-based evaluation by race/sex/age. Each part prints concise explanations of the numeric results so you can run the notebook and read the interpretation immediately.

In [ ]:
# ============================================================================
# HELPER FUNCTIONS FOR COMPREHENSIVE ANALYSIS
# ============================================================================

import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics.pairwise import rbf_kernel


def _to_dense(x):
    """Convert sparse arrays to dense NumPy arrays."""
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)


def psi_numeric(train_values, test_values, bins=10, eps=1e-6):
    """
    Population Stability Index (PSI) using train-derived bins.
    
    PSI measures how much a distribution has shifted between train and test.
    Higher PSI indicates larger distributional drift.
    - PSI < 0.1: minimal drift
    - PSI 0.1-0.25: small drift (monitor)
    - PSI > 0.25: significant drift (investigate)
    """
    train_values = pd.to_numeric(pd.Series(train_values), errors="coerce").dropna().values
    test_values = pd.to_numeric(pd.Series(test_values), errors="coerce").dropna().values

    if len(train_values) == 0 or len(test_values) == 0:
        return np.nan

    # Quantile bins derived from training distribution
    quantiles = np.linspace(0, 1, bins + 1)
    cut_points = np.unique(np.quantile(train_values, quantiles))

    # Fallback if quantiles collapse
    if len(cut_points) < 3:
        lo = min(train_values.min(), test_values.min())
        hi = max(train_values.max(), test_values.max())
        if lo == hi:
            return 0.0
        cut_points = np.linspace(lo, hi, bins + 1)

    cut_points[0] = -np.inf
    cut_points[-1] = np.inf

    train_counts, _ = np.histogram(train_values, bins=cut_points)
    test_counts, _ = np.histogram(test_values, bins=cut_points)

    train_pct = np.clip(train_counts / train_counts.sum(), eps, None)
    test_pct = np.clip(test_counts / test_counts.sum(), eps, None)

    return np.sum((train_pct - test_pct) * np.log(train_pct / test_pct))


def mmd_rbf(X_a, X_b, gamma=None, max_n=500, random_state=42):
    """
    Unbiased Maximum Mean Discrepancy (MMD^2) with RBF kernel.
    
    MMD is a kernel-based test for distributional equality in high dimensions.
    Larger MMD indicates greater differences between distributions.
    Works on both raw features and encoded feature spaces.
    """
    rng = np.random.default_rng(random_state)

    X_a = _to_dense(X_a)
    X_b = _to_dense(X_b)

    if X_a.shape[0] > max_n:
        idx_a = rng.choice(X_a.shape[0], size=max_n, replace=False)
        X_a = X_a[idx_a]
    if X_b.shape[0] > max_n:
        idx_b = rng.choice(X_b.shape[0], size=max_n, replace=False)
        X_b = X_b[idx_b]

    if gamma is None:
        gamma = 1.0 / X_a.shape[1]

    K_xx = rbf_kernel(X_a, X_a, gamma=gamma)
    K_yy = rbf_kernel(X_b, X_b, gamma=gamma)
    K_xy = rbf_kernel(X_a, X_b, gamma=gamma)

    np.fill_diagonal(K_xx, 0.0)
    np.fill_diagonal(K_yy, 0.0)

    m = X_a.shape[0]
    n = X_b.shape[0]

    term_xx = K_xx.sum() / (m * (m - 1))
    term_yy = K_yy.sum() / (n * (n - 1))
    term_xy = 2.0 * K_xy.mean()

    return term_xx + term_yy - term_xy


def evaluate_classifier(model, X, y, label):
    """
    Comprehensive train/test performance summary.
    Returns a dictionary of key metrics used for overfitting diagnosis.
    """
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    return {
        "model": label,
        "n": len(y),
        "accuracy": accuracy_score(y, y_pred),
        "auc": roc_auc_score(y, y_prob),
        "brier": brier_score_loss(y, y_prob),
        "logloss": log_loss(y, y_prob),
        "positive_rate_pred": y_pred.mean(),
        "mean_score": y_prob.mean(),
    }


def permutation_importance_table(model, X, y, scoring="roc_auc", n_repeats=15, random_state=42):
    """
    Compute permutation importance on raw pipeline inputs.
    Identifies features whose degradation hurts model performance most.
    """
    pi = permutation_importance(
        model,
        X,
        y,
        scoring=scoring,
        n_repeats=n_repeats,
        random_state=random_state,
    )

    out = pd.DataFrame({
        "feature": X.columns,
        "importance_mean": pi.importances_mean,
        "importance_std": pi.importances_std,
    }).sort_values("importance_mean", ascending=False)

    return out.reset_index(drop=True)


def pairwise_swap_shift(model, X, feature_col, value_a, value_b):
    """
    Counterfactual sensitivity test:
    Swap attribute values (A <-> B) and measure mean absolute change in predicted probability.
    
    Large shifts suggest the feature has spurious/causal influence on predictions.
    Used to detect if protected attributes are indirectly influencing risk scores.
    """
    work = X.copy()
    mask = work[feature_col].astype(str).isin([value_a, value_b])

    if mask.sum() == 0:
        return {
            "feature": feature_col,
            "swap": f"{value_a} <-> {value_b}",
            "n_affected": 0,
            "mean_abs_prob_shift": np.nan,
        }

    base_prob = model.predict_proba(work.loc[mask])[:, 1]

    cf = work.loc[mask].copy()
    swapped = cf[feature_col].astype(str).map({value_a: value_b, value_b: value_a})

    cf.loc[:, feature_col] = swapped.values
    cf_prob = model.predict_proba(cf)[:, 1]

    return {
        "feature": feature_col,
        "swap": f"{value_a} <-> {value_b}",
        "n_affected": int(mask.sum()),
        "mean_abs_prob_shift": float(np.mean(np.abs(cf_prob - base_prob))),
    }


def slice_metrics(model, X, y, group_col):
    """
    Slice-based evaluation: compute accuracy, AUC, FPR, FNR per group.
    Used to detect performance disparities across demographic groups.
    """
    pred_prob = model.predict_proba(X)[:, 1]
    pred = (pred_prob >= 0.5).astype(int)

    eval_df = X[[group_col]].copy()
    eval_df["actual"] = y.values
    eval_df["pred"] = pred
    eval_df["pred_prob"] = pred_prob

    rows = []
    for grp, g in eval_df.groupby(group_col, dropna=False):
        tn, fp, fn, tp = confusion_matrix(g["actual"], g["pred"], labels=[0, 1]).ravel()

        auc = roc_auc_score(g["actual"], g["pred_prob"]) if g["actual"].nunique() > 1 else np.nan
        brier = brier_score_loss(g["actual"], g["pred_prob"])

        rows.append({
            "slice_feature": group_col,
            "slice_value": grp,
            "n": len(g),
            "accuracy": accuracy_score(g["actual"], g["pred"]),
            "auc": auc,
            "brier": brier,
            "fpr": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
            "fnr": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
            "positive_rate_pred": g["pred"].mean(),
            "mean_score": g["pred_prob"].mean(),
        })

    return pd.DataFrame(rows)


def stress_test_priors(model, X, deltas=(0, 2, 5, 10)):
    """
    Stress test: add a delta to priors_count and observe outcome shift.
    Measures model sensitivity to this criminogenic factor.
    """
    rows = []
    base_min = X["priors_count"].min()
    base_max = X["priors_count"].max()

    for d in deltas:
        X_s = X.copy()
        X_s["priors_count"] = np.clip(X_s["priors_count"] + d, base_min, base_max)

        prob = model.predict_proba(X_s)[:, 1]
        pred = (prob >= 0.5).astype(int)

        rows.append({
            "delta_priors_count": d,
            "mean_pred_prob": prob.mean(),
            "median_pred_prob": np.median(prob),
            "share_pred_high_risk": pred.mean(),
        })

    return pd.DataFrame(rows)


def plot_ice_numeric(model, X, feature_col, values, n_instances=6, random_state=42, title=None):
    """
    Individual Conditional Expectation (ICE) plot for one numeric feature.
    Shows how model predictions change for selected individuals as feature varies.
    """
    rng = np.random.default_rng(random_state)
    idx = rng.choice(X.index, size=min(n_instances, len(X)), replace=False)

    plt.figure(figsize=(8, 5))
    for i in idx:
        row = X.loc[[i]].copy()
        preds = []
        for v in values:
            temp = row.copy()
            temp[feature_col] = v
            preds.append(model.predict_proba(temp)[:, 1][0])

        plt.plot(values, preds, alpha=0.7)

    plt.xlabel(feature_col)
    plt.ylabel("Predicted probability (high risk)")
    plt.title(title if title else f"ICE curves for {feature_col}")
    plt.show()


def global_sensitivity_index(model, X, feature_col, values):
    """
    Approximate global sensitivity: variance of average predictions as feature is swept.
    Higher variance indicates stronger model sensitivity to that feature.
    """
    mean_preds = []

    for v in values:
        X_s = X.copy()
        X_s[feature_col] = v
        prob = model.predict_proba(X_s)[:, 1]
        mean_preds.append(prob.mean())

    mean_preds = np.array(mean_preds)

    return pd.DataFrame({
        "feature": [feature_col],
        "sensitivity_index": [np.var(mean_preds)],
        "min_mean_score": [mean_preds.min()],
        "max_mean_score": [mean_preds.max()],
        "range_mean_score": [mean_preds.max() - mean_preds.min()],
    })

print("✓ Helper functions loaded successfully.")

In [ ]:
# ============================================================================
# PART B: DISTRIBUTION DRIFT
# ============================================================================
# Distribution drift occurs when the training and test data have different
# underlying distributions. This can lead to model performance degradation.

print("\n" + "="*70)
print("PART B: INPUT & SCORE DISTRIBUTION DRIFT")
print("="*70)

# 1) Input drift on raw numeric features: PSI + KS
print("\n--- B.1: Input Distribution Drift (Numeric Features) ---")
print("PSI (Population Stability Index): Measures shift in univariate distributions")
print("KS (Kolmogorov-Smirnov): Non-parametric test for distributional equality\n")

drift_rows = []
for col in numeric_features:
    psi_val = psi_numeric(X_train[col], X_test[col], bins=10)
    ks_stat, ks_p = ks_2samp(X_train[col], X_test[col])

    drift_rows.append({
        "feature": col,
        "train_mean": X_train[col].mean(),
        "test_mean": X_test[col].mean(),
        "PSI": psi_val,
        "KS_stat": ks_stat,
        "KS_pvalue": ks_p,
    })

input_drift_table = pd.DataFrame(drift_rows).sort_values("PSI", ascending=False)
print(input_drift_table.round(4).to_string(index=False))

print("\n💡 Interpretation:")
print("  • PSI < 0.1: minimal drift (stable)")
print("  • PSI 0.1-0.25: monitor for performance decay")
print("  • PSI > 0.25: significant drift (investigate immediately)")
print("  • KS p-value < 0.05: statistically significant distributional shift")

# 2) Global high-dimensional drift: MMD on encoded inputs
print("\n--- B.2: Global High-Dimensional Drift (Encoded Feature Space) ---")
print("MMD: Maximum Mean Discrepancy measures difference in distributions")
print("     in high-dimensional feature spaces\n")

fitted_preprocessor = lr_pipeline.named_steps["preprocessor"]
X_train_enc = fitted_preprocessor.transform(X_train)
X_test_enc = fitted_preprocessor.transform(X_test)

mmd_val = mmd_rbf(X_train_enc, X_test_enc, gamma=None, max_n=500, random_state=42)
print(f"MMD^2(train, test) = {mmd_val:.6f}")

print("\n💡 Interpretation:")
print(f"  • MMD measures global shift across all {X_train_enc.shape[1]} encoded features")
print(f"  • Larger MMD indicates more pronounced distributional mismatch")
print(f"  • Current MMD: {mmd_val:.6f} (compare to univariate drifts above)")

# 3) Score drift: train vs test predicted probabilities
print("\n--- B.3: Score Distribution Drift (Predicted Probabilities) ---")
print("Compares distributions of model-predicted scores (not labels)\n")

score_drift_rows = []
models = {
    "Logistic Regression": lr_pipeline,
    "Gradient-Boosted Tree": gbt_pipeline,
}

for name, model in models.items():
    train_prob = model.predict_proba(X_train)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    psi_score = psi_numeric(train_prob, test_prob, bins=10)
    ks_stat, ks_p = ks_2samp(train_prob, test_prob)

    score_drift_rows.append({
        "model": name,
        "train_mean_score": train_prob.mean(),
        "test_mean_score": test_prob.mean(),
        "PSI_score": psi_score,
        "KS_stat_score": ks_stat,
        "KS_pvalue_score": ks_p,
    })

score_drift_table = pd.DataFrame(score_drift_rows)
print(score_drift_table.round(4).to_string(index=False))

print("\n💡 Interpretation:")
print("  • Large shift in mean score suggests input drift propagates to predictions")
print("  • PSI_score: drift in predicted probability distributions (recalibration signals)")
print("  • Action: if PSI_score > 0.25 for any model, trigger retraining or threshold adjustment")
print("="*70)

In [ ]:
# ============================================================================
# PART C: GENERALIZATION & OVERFITTING DIAGNOSIS
# ============================================================================
# Generalization measures how well a model trained on data performs on unseen data.
# Large train-test gaps indicate overfitting (model memorizes noise).

print("\n" + "="*70)
print("PART C: GENERALIZATION DIAGNOSTICS & OVERFITTING ANALYSIS")
print("="*70)

print("\n--- C.1: Train vs. Test Performance Metrics ---")
print("Metrics: Accuracy, AUC (discrimination), Brier (calibration), LogLoss\n")

gen_rows = []
for name, model in models.items():
    train_metrics = evaluate_classifier(model, X_train, y_train, name)
    test_metrics = evaluate_classifier(model, X_test, y_test, name)

    gen_rows.append({
        "model": name,
        "train_accuracy": train_metrics["accuracy"],
        "test_accuracy": test_metrics["accuracy"],
        "accuracy_gap": train_metrics["accuracy"] - test_metrics["accuracy"],
        "train_auc": train_metrics["auc"],
        "test_auc": test_metrics["auc"],
        "auc_gap": train_metrics["auc"] - test_metrics["auc"],
        "train_brier": train_metrics["brier"],
        "test_brier": test_metrics["brier"],
        "brier_gap": train_metrics["brier"] - test_metrics["brier"],
        "train_logloss": train_metrics["logloss"],
        "test_logloss": test_metrics["logloss"],
        "logloss_gap": train_metrics["logloss"] - test_metrics["logloss"],
    })

generalization_table = pd.DataFrame(gen_rows)
print(generalization_table.round(4).to_string(index=False))

print("\n💡 Overfitting Diagnosis:")
print("  • Accuracy_gap > 0.05: model overfitting (watch for noise reliance)")
print("  • AUC_gap > 0.03: discrimination power not generalizing well")
print("  • Brier_gap > 0.02: model poorly calibrated on test (predictions unreliable)")
print("  • LogLoss_gap: probabilistic loss gap (lower is better)")
print("  • Action: if gaps large, regularize model, reduce complexity, or collect more data")

# Permutation importance: compare train vs. test
print("\n--- C.2: Permutation Importance (Train vs. Test) ---")
print("Identifies features whose removal hurts model performance most.")
print("Large divergence between train & test importance suggests overfitting to train-specific signal\n")

for name, model in models.items():
    print(f"\n{name} — Top 10 Features")
    print("-" * 60)
    
    pi_train = permutation_importance_table(model, X_train, y_train, scoring="roc_auc")
    pi_test = permutation_importance_table(model, X_test, y_test, scoring="roc_auc")

    # Merge and compare
    merged = pi_train.head(10)[["feature", "importance_mean"]].rename(
        columns={"importance_mean": "train_importance"}
    ).merge(
        pi_test[["feature", "importance_mean"]].rename(
            columns={"importance_mean": "test_importance"}
        ),
        on="feature",
        how="outer"
    ).fillna(0)
    
    merged["importance_divergence"] = merged["train_importance"] - merged["test_importance"]
    merged = merged.sort_values("train_importance", ascending=False).head(10)
    
    print(merged.round(4).to_string(index=False))
    
    print("\n💡 Interpretation:")
    print("   • High train_importance but low test_importance = overfitting signal")
    print("   • Stable importances across train/test = generalizable signal")

print("\n" + "="*70)

In [ ]:
# ============================================================================
# PART D: SPURIOUS-CORRELATION PROBE (Counterfactual Swaps)
# ============================================================================
# Detects if protected/sensitive attributes have spurious influence on predictions.
# If swapping (e.g., race) significantly changes score, the model is using that
# attribute (directly or as a proxy) in ways that may violate fairness principles.

print("\n" + "="*70)
print("PART D: SPURIOUS-CORRELATION PROBE (Counterfactual Analysis)")
print("="*70)

print("\nMethod: Swap protected attribute (e.g., race, gender) for test individuals")
print("        and measure mean absolute shift in predicted probabilities.")
print("        Large shifts indicate spurious or direct dependency.\n")

# Define swaps to test: (column, value_a, value_b)
swap_specs = [
    ("race_factor", "African-American", "Caucasian"),
    ("gender_factor", "Female", "Male"),
    ("crime_factor", "F", "M"),  # Felony vs. Misdemeanor
]

for name, model in models.items():
    print(f"\n{name.upper()}")
    print("-" * 60)
    
    shifts = []
    for feature_col, a, b in swap_specs:
        # Skip if column not in data
        if feature_col not in X_test.columns:
            continue
        
        result = pairwise_swap_shift(model, X_test, feature_col, a, b)
        shifts.append(result)

    if shifts:
        shift_table = pd.DataFrame(shifts)
        print(shift_table.round(4).to_string(index=False))

        print("\n💡 Interpretation:")
        print("  • mean_abs_prob_shift > 0.05: attribute has substantial influence on predictions")
        print("  • mean_abs_prob_shift > 0.10: likely spurious/unfair dependence (ACTION REQUIRED)")
        print("  • n_affected: number of individuals whose predicted score would change")
        print("\n⚠️  What this tells us:")
        print("     – Large shifts for protected attributes (race, gender) signal potential bias")
        print("     – Model may be using attribute directly or as proxy for unmeasured factors")
        print("     – May violate fairness criterion: 'score should not depend on protected attribute'")
        print("\n✓ What actions are justified:")
        print("    1. If shift > 0.10 for race/gender: audit feature engineering & interactions")
        print("    2. Consider regularization or fairness constraints (e.g., demographic parity)")
        print("    3. Retrain without proxies; validate on fairness metrics (FPR parity, EOppGap)")

print("\n" + "="*70)

In [ ]:
# ============================================================================
# PART E: ROBUSTNESS (Stress Testing & Sensitivity Analysis)
# ============================================================================
# Stress tests measure how model predictions respond to adverse or counterfactual scenarios.
# ICE (Individual Conditional Expectation) plots show per-individual sensitivity curves.
# Global sensitivity index aggregates across all individuals.

print("\n" + "="*70)
print("PART E: ROBUSTNESS VIA STRESS TESTING & SENSITIVITY ANALYSIS")
print("="*70)

print("\n--- E.1: Stress Test on priors_count ---")
print("Scenario: Add δ to priors_count, observe mean predicted probability shift.")
print("Measures sensitivity to historical criminogenic factor (prior arrests).\n")

for name, model in models.items():
    print(f"\n{name.upper()}")
    print("-" * 60)
    
    stress_table = stress_test_priors(model, X_test, deltas=(0, 2, 5, 10))
    print(stress_table.round(4).to_string(index=False))

    print("\n💡 Interpretation:")
    print("  • share_pred_high_risk increases with δ: model increases risk prediction when priors go up")
    print("  • sharp slope (Δshare >> 10% per increment): model heavily weights priors (expected)")
    print("  • flat/absent slope: priors_count may be underutilized (model ignores signal)")
    print("\n✓ What actions are justified:")
    print("   – If slope too steep: priors dominates; risk of perpetuating recidivism cycles")
    print("   – Consider regularization or fairness overlay (e.g., cap contribution of priors)")
    print("   – Compare to LR vs. GBT: tree may be more sensitive to non-linear interactions")

# ICE-style sensitivity plots for priors_count
print("\n\n--- E.2: Individual Conditional Expectation (ICE) Curves ---")
print("Shows predicted probability for each individual as priors_count varies.\n")

priors_grid = np.arange(
    int(X_test["priors_count"].min()),
    int(X_test["priors_count"].max()) + 1
)

plot_ice_numeric(
    lr_pipeline,
    X_test,
    feature_col="priors_count",
    values=priors_grid,
    n_instances=6,
    random_state=42,
    title="Logistic Regression — ICE Curves for priors_count"
)

plot_ice_numeric(
    gbt_pipeline,
    X_test,
    feature_col="priors_count",
    values=priors_grid,
    n_instances=6,
    random_state=42,
    title="Gradient-Boosted Tree — ICE Curves for priors_count"
)

print("\n💡 Interpretation of ICE plots:")
print("  • Parallel lines: uniform (monotonic) sensitivity across all individuals")
print("  • Crossing/divergent lines: interaction effects (priors impact depends on other features)")
print("  • Steep slopes: strong sensitivity to priors_count")
print("  • Flat slopes: weak sensitivity (may indicate other features dominate)")

# Global sensitivity index for priors_count
print("\n--- E.3: Global Sensitivity Index (priors_count) ---")
print("Aggregated measure: variance of mean predictions across priors_count range.\n")

lr_sens = global_sensitivity_index(lr_pipeline, X_test, "priors_count", priors_grid)
gbt_sens = global_sensitivity_index(gbt_pipeline, X_test, "priors_count", priors_grid)

sensitivity_table = pd.concat([
    lr_sens.assign(model="Logistic Regression"),
    gbt_sens.assign(model="Gradient-Boosted Tree"),
], ignore_index=True)

print(sensitivity_table.round(6).to_string(index=False))

print("\n💡 Interpretation:")
print("  • sensitivity_index: variance of mean scores as feature varies")
print("  • range_mean_score: difference between max and min predicted probabilities")
print("  • Higher range = model is more sensitive to feature variations")
print("  • Useful for: prioritizing which features most impact predictions (policy lever)")

print("\n" + "="*70)

In [ ]:
# ============================================================================
# PART A: SLICE-BASED EVALUATION (Fairness Analysis by Demographic Groups)
# ============================================================================
# Slice-based evaluation breaks performance down by demographic subgroups
# (race, gender, age, crime type) to detect disparities.
# 
# Key fairness metrics:
# - Accuracy: overall correctness per group (should be similar across groups)
# - FPR (False Positive Rate): false alarms; high FPR on minorities = bias
# - FNR (False Negative Rate): missed cases; high FNR = under-prediction for group
# - AUC: discrimination power per group (should not diverge wildly)

print("\n" + "="*70)
print("PART A: SLICE-BASED FAIRNESS EVALUATION")
print("="*70)

slice_features = ["race_factor", "gender_factor", "age_factor", "crime_factor"]

for name, model in models.items():
    print(f"\n{name.upper()}")
    print("="*70)
    
    slice_tables = []
    for col in slice_features:
        # Skip if column not in test set
        if col not in X_test.columns:
            continue
        
        result = slice_metrics(model, X_test, y_test, group_col=col)
        if len(result) > 0:
            slice_tables.append(result)
    
    if slice_tables:
        slice_eval = pd.concat(slice_tables, ignore_index=True)
        slice_eval = slice_eval.sort_values(["slice_feature", "n"], ascending=[True, False])
        
        print("\nDetailed Slice Metrics:")
        print(slice_eval.round(4).to_string(index=False))

        # Compute disparity statistics
        print("\n--- Fairness Disparity Analysis ---")
        for feature in slice_eval["slice_feature"].unique():
            feature_data = slice_eval[slice_eval["slice_feature"] == feature]
            
            print(f"\n{feature}:")
            accuracy_range = feature_data["accuracy"].max() - feature_data["accuracy"].min()
            fpr_range = feature_data["fpr"].max() - feature_data["fpr"].min()
            fnr_range = feature_data["fnr"].max() - feature_data["fnr"].min()
            auc_range = feature_data["auc"].max() - feature_data["auc"].min()
            
            print(f"  Accuracy range: {accuracy_range:.4f}")
            print(f"  FPR range:      {fpr_range:.4f} (false alarm disparity)")
            print(f"  FNR range:      {fnr_range:.4f} (miss rate disparity)")
            print(f"  AUC range:      {auc_range:.4f}")

        print("\n What each metric tells us:")
        print("  • Accuracy: Overall correctness (should be similar across groups)")
        print("  • AUC: Discrimination power (ability to rank high-risk correctly)")
        print("  • FPR: False positive rate (labeling innocents as 'high risk')")
        print("  • FNR: False negative rate (failing to flag actual recidivists)")
        print("  • positive_rate_pred: % predicted as high-risk (selection rate)")
        print("  • mean_score: average predicted probability per group")

        print("\n  What disparities mean:")
        print("  • High FPR on minority group: more false alarms → bias in algorithm")
        print("  • High FNR on minority group: under-prediction → systematic under-flagging")
        print("  • Different accuracy: model learns different patterns for different groups")
        print("  • Different mean_score: groups receive systematically different risk scores")

        print("\n✓ What actions are justified:")
        print("   1. If accuracy_range > 0.10: group-specific model tuning may be needed")
        print("   2. If FPR_range > 0.15: audit for bias; consider fairness constraint")
        print("   3. If FNR_range > 0.15: model under-flags certain groups (adjust threshold/threshold per group)")
        print("   4. Publish these metrics: transparency on fairness trade-offs")
        print("   5. Consider: equalized odds (same FPR/FNR across groups)")
        print("   6. Consider: demographic parity (same selection rate across groups)")

print("\n" + "="*70)
print("END OF ANALYSIS")
print("="*70)
print("\n Summary of insights:")
print("   • Distribution Drift (Part B): input/score stability across train/test")
print("   • Generalization (Part C): train/test gap diagnosis + feature importance stability")
print("   • Spurious Correlations (Part D): counterfactual sensitivity to protected attributes")
print("   • Robustness (Part E): stress-test sensitivity curves and global importance")
print("   • Fairness (Part A): group-level performance disparities in accuracy, FPR, FNR")
print("\n Recommended audit checklist:")
print("   ☐ Input drift PSI < 0.25 for all numeric features")
print("   ☐ Train-test AUC gap < 0.03 (generalization OK)")
print("   ☐ Permutation importance stable train vs. test")
print("   ☐ Counterfactual race/gender swap < 0.05 mean shift")
print("   ☐ Priors_count stress test shows expected monotonic increase")
print("   ☐ Slice-based FPR/FNR ranges < 0.15 across all demographic groups")
print("   ☐ No statistically significant performance gap in any slice")
print("="*70)


PART A: SLICE-BASED FAIRNESS EVALUATION


NameError: name 'models' is not defined